# 09 — Exceptions, Debugging, and Logging

Goal: handle failures deliberately, debug efficiently, and produce useful logs (instead of print spam).

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: Exceptions as control flow (sparingly)

Exceptions represent exceptional situations (invalid input, I/O failures, etc).
Use them to report errors, not to hide bugs.

Key patterns:
- catch specific exceptions
- re-raise with context
- `finally` for cleanup

In [ ]:

def safe_int(s: str) -> int | None:
    try:
        return int(s)
    except ValueError:
        return None

print(safe_int("12"))
print(safe_int("nope"))


## 2.
L2: Custom exceptions (domain errors)

Custom exception types make error handling cleaner.

In [ ]:

class ConfigError(Exception):
    """Raised when configuration is invalid."""
    pass

def parse_port(s: str) -> int:
    try:
        port = int(s)
    except ValueError as e:
        raise ConfigError(f"port must be int, got {s!r}") from e
    if not (1 <= port <= 65535):
        raise ConfigError(f"port out of range: {port}")
    return port

try:
    parse_port("99999")
except ConfigError as e:
    print("ConfigError:", e)


## 3.
L3: Assertions vs exceptions

- `assert` is for **developer invariants** (“this must be true if code is correct”).
- Don’t use `assert` for user input validation in production (asserts can be disabled with `-O`).

In [ ]:

def mean(xs: list[float]) -> float:
    assert len(xs) > 0, "cannot compute mean of empty list"
    return sum(xs) / len(xs)

print(mean([1,2,3]))


## 4.
L4: Tracebacks and debugging

Tools:
- `traceback` module to capture/format
- `pdb` / `breakpoint()` for interactive stepping
- `rich` (third-party) for pretty tracebacks (optional)

In [ ]:

import traceback

def boom():
    return 1 / 0

try:
    boom()
except Exception:
    print("formatted traceback (tail):")
    tail = traceback.format_exc().splitlines()[-3:]
    print("\n".join(tail))


## 5.
L5: Logging basics (stdlib)

Logging is hierarchical by name.

Guidelines:
- Create a module-level logger: `logger = logging.getLogger(__name__)`
- Use levels: DEBUG/INFO/WARNING/ERROR/CRITICAL
- Avoid logging secrets/PII

In [ ]:

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)

logger = logging.getLogger("demo")
logger.info("hello")
logger.warning("something might be off")


## 6.
L6: Exercises

1. Write `retry(fn, attempts=3)` that retries a callable on exceptions.
2. Create a custom exception for a domain error you care about.
3. Configure logging to also write to a file `app.log`.

In [ ]:

from collections.abc import Callable

def retry(fn: Callable[[], object], attempts: int = 3):
    last: Exception | None = None
    for _ in range(attempts):
        try:
            return fn()
        except Exception as e:
            last = e
    raise RuntimeError(f"failed after {attempts} attempts") from last

i = 0
def flaky():
    global i
    i += 1
    if i < 2:
        raise ValueError("nope")
    return "ok"

print(retry(flaky, attempts=3))


## 7.
L7: Exception chaining (`raise ... from ...`)

Chaining keeps the original exception as context.
Use it when converting low-level errors to domain errors.

In [ ]:

def read_int(s: str) -> int:
    try:
        return int(s)
    except ValueError as e:
        raise RuntimeError("failed to parse int") from e

try:
    read_int("nope")
except Exception as e:
    print(type(e).__name__, "->", type(e.__cause__).__name__)


## 8.
L8: `warnings` (non-fatal problems)

Warnings are for “this is suspicious but not necessarily fatal”.
Libraries often emit DeprecationWarnings; you can filter them.

In [ ]:

import warnings

warnings.warn("demo warning", UserWarning)
print("continued execution")


## 9.
L9: `contextlib.suppress` and `ExitStack`

`contextlib.suppress` is a clean way to ignore *specific* exceptions.

`ExitStack` lets you manage a dynamic number of context managers.

In [ ]:

from contextlib import suppress, ExitStack
from pathlib import Path

p = Path("no_such_file.txt")
with suppress(FileNotFoundError):
    p.unlink()

with ExitStack() as stack:
    files = [stack.enter_context(Path(f"t{i}.txt").open("w", encoding="utf-8")) for i in range(3)]
    for i, f in enumerate(files):
        f.write(f"file {i}\n")

for i in range(3):
    Path(f"t{i}.txt").unlink(missing_ok=True)
print("ok")
